# TP 1 — La naturaleza de la econometría y los datos económicos

**Materia:** Econometría I
**Referencia:** Wooldridge, *Introducción a la econometría*, capítulo 1
**Entregable:** Ejercicios 1.2, 1.5, 1.6, 1.7 y 1.8

**Nota sobre el código:** cada ejercicio con cálculos se muestra primero en R
(bloque de código, no ejecutado en este notebook) y luego en Python (celda
ejecutada, con la salida real debajo). Los datos provienen del paquete
`wooldridge`, disponible tanto en R (`library(wooldridge)`) como en Python
(`import wooldridge as woo`).

In [1]:
# Configuración (Python)
import wooldridge as woo
import numpy as np
import pandas as pd

pd.set_option('display.precision', 4)


---
## Ejercicio 1.2 (Wooldridge C1.2, `bwght`) — Nivel R

**Enunciado.** Informe cuántas mujeres hay en la muestra, cuántas fumaron
durante el embarazo y la cantidad promedio de cigarrillos diarios (`cigs`).
Luego calcule ese promedio solo entre las fumadoras. Explique por qué los dos
números difieren tanto y cuál de los dos responde mejor a la pregunta "cuánto
fuma una mujer que fuma durante el embarazo".

**Código en R**

```r
library(wooldridge)
data(bwght, package = "wooldridge")

nrow(bwght)                          # tamaño de la muestra (todas mujeres que dieron a luz)
sum(bwght$cigs > 0)                  # cuántas fumaron
mean(bwght$cigs)                     # promedio sobre toda la muestra
mean(bwght$cigs[bwght$cigs > 0])     # promedio solo entre fumadoras
```

In [2]:
# Código en Python
bwght = woo.dataWoo('bwght')

n = len(bwght)
fumadoras = (bwght['cigs'] > 0).sum()
media_todas = bwght['cigs'].mean()
media_fumadoras = bwght.loc[bwght['cigs'] > 0, 'cigs'].mean()

print('Tamaño de la muestra (mujeres):', n)
print('Mujeres que fumaron durante el embarazo:', fumadoras)
print('Promedio de cigs, toda la muestra:', round(media_todas, 3))
print('Promedio de cigs, solo fumadoras:', round(media_fumadoras, 3))


Tamaño de la muestra (mujeres): 1388
Mujeres que fumaron durante el embarazo: 212
Promedio de cigs, toda la muestra: 2.087
Promedio de cigs, solo fumadoras: 13.665


**Resultados.** La muestra tiene 1,388 mujeres que dieron a luz. De ellas,
212 (un 15.3%) reportan haber fumado durante el embarazo. El promedio de
cigarrillos diarios sobre toda la muestra es 2.09, pero calculado solo entre
las fumadoras sube a 13.67.

**Interpretación.** Los dos números difieren tanto porque el primero promedia
sobre una variable con una masa enorme de ceros: el 84.7% de la muestra no
fumó, así que ese 2.09 mezcla a quienes fuman mucho con quienes no fuman nada
y termina sin describir bien a ningún subgrupo. El segundo número, calculado
condicional en `cigs > 0`, sí aísla el comportamiento del grupo relevante y
por eso responde mejor a "cuánto fuma una mujer que fuma": 13.67
cigarrillos diarios es una intensidad de consumo real, mientras que 2.09 no
es la intensidad de nadie en particular. Aun así, conviene sospechar del
2.09 y del 13.67 por igual: fumar durante el embarazo está estigmatizado
socialmente, por lo que el autorreporte probablemente subestime la
prevalencia y la intensidad verdaderas.

---
## Ejercicio 1.5 (`wage1`) — Nivel V

**Enunciado.** Calcule el salario promedio por hora de varones y mujeres.
Nombre tres variables presentes en el dataset que podrían explicar parte de
la brecha sin invocar discriminación y explique en qué dirección operaría
cada una.

**Código en R**

```r
library(wooldridge)
data(wage1, package = "wooldridge")

aggregate(wage ~ female, data = wage1, FUN = mean)
aggregate(cbind(educ, exper, tenure, profocc, clerocc, servocc) ~ female,
          data = wage1, FUN = mean)
```

In [3]:
# Código en Python
wage1 = woo.dataWoo('wage1')

salario_por_sexo = wage1.groupby('female')['wage'].mean()
print(salario_por_sexo)

brecha_abs = salario_por_sexo[0] - salario_por_sexo[1]
brecha_pct = 100 * brecha_abs / salario_por_sexo[0]
print(f'\nBrecha: {brecha_abs:.3f} USD/hora ({brecha_pct:.1f}% del salario de los varones)')

print()
print(wage1.groupby('female')[['educ', 'exper', 'tenure']].mean())
print()
print(wage1.groupby('female')[['profocc', 'clerocc', 'servocc']].mean())


female
0    7.0995
1    4.5877
Name: wage, dtype: float64

Brecha: 2.512 USD/hora (35.4% del salario de los varones)

           educ    exper  tenure
female                          
0       12.7883  17.5584  6.4745
1       12.3175  16.4286  3.6151

        profocc  clerocc  servocc
female                           
0        0.4489   0.0401   0.0876
1        0.2778   0.3056   0.1984


**Resultados.** El salario promedio es 7.10 dólares/hora para los varones y
4.59 dólares/hora para las mujeres: una brecha de 2.51 dólares/hora, un 35.4%
del salario masculino.

**Interpretación.** Sin invocar discriminación, tres variables de `wage1`
podrían explicar parte de esta brecha:

1. **`tenure` (antigüedad en el empleo actual).** Los varones promedian 6.47
   años de antigüedad contra 3.62 de las mujeres. Como el salario suele crecer
   con la antigüedad (capital específico a la empresa, mejores condiciones
   negociadas con el tiempo), una antigüedad más baja empuja el salario
   femenino promedio hacia abajo, independientemente de cualquier trato
   diferencial.
2. **Distribución ocupacional (`profocc`, `clerocc`, `servocc`).** El 44.9%
   de los varones está en ocupaciones profesionales, contra 27.8% de las
   mujeres, mientras que ellas están sobrerrepresentadas en ocupaciones
   administrativas (30.6% vs. 4.0%) y de servicios (19.8% vs. 8.8%). Si estas
   ocupaciones difieren en productividad marginal o en la escasez relativa de
   la mano de obra que demandan, la brecha salarial reflejaría en parte esta
   segregación ocupacional y no un trato distinto dentro de la misma
   ocupación.
3. **`exper` (experiencia potencial).** Los varones promedian 17.56 años de
   experiencia contra 16.43 de las mujeres; la diferencia es menor que en
   antigüedad, pero opera en la misma dirección: menos experiencia acumulada
   se traduce, vía retornos positivos a la experiencia, en un salario más
   bajo.

Ninguna de estas tres variables agota la explicación ni descarta
discriminación: que operen "sin invocarla" no significa que la discriminación
esté ausente, solo que hay canales alternativos y observables que también
predicen parte de la brecha y que deberían controlarse antes de atribuir el
residuo a un trato diferencial.

---
## Ejercicio 1.6 (aproximación logarítmica) — Nivel V

**Enunciado.** La escuela A gasta 6,000 dólares por alumno y la escuela B
gasta 5,500. Calcule la diferencia porcentual exacta y la aproximación
logarítmica. Repita con 5,500 contra 11,000. ¿Qué le pasa a la aproximación y
en qué dirección se equivoca? Formule una regla práctica sobre cuándo
usarla.

**Código en R**

```r
cambio <- function(x0, x1) {
  exacto <- 100 * (x1 - x0) / x0
  aproximado <- 100 * (log(x1) - log(x0))
  c(exacto = exacto, aproximado = aproximado, error = aproximado - exacto)
}

cambio(5500, 6000)    # de B a A
cambio(5500, 11000)   # de 5,500 a 11,000
```

In [4]:
# Código en Python
def cambio(x0, x1):
    exacto = 100 * (x1 - x0) / x0
    aproximado = 100 * (np.log(x1) - np.log(x0))
    return exacto, aproximado, aproximado - exacto

for x0, x1, etiqueta in [(5500, 6000, 'B -> A (5,500 a 6,000)'),
                          (5500, 11000, '5,500 a 11,000')]:
    exacto, aproximado, error = cambio(x0, x1)
    print(f'{etiqueta}: exacto={exacto:.2f}%, log={aproximado:.2f}%, error={error:.2f} p.p.')


B -> A (5,500 a 6,000): exacto=9.09%, log=8.70%, error=-0.39 p.p.
5,500 a 11,000: exacto=100.00%, log=69.31%, error=-30.69 p.p.


**Resultados.** Para el cambio de 5,500 a 6,000 dólares, el cambio exacto es
9.09% y la aproximación logarítmica da 8.70%: un error pequeño, de menos de
medio punto porcentual. Para el cambio de 5,500 a 11,000, el cambio exacto es
100% y la aproximación logarítmica da apenas 69.31%: un error de casi 31
puntos porcentuales.

**Interpretación.** La aproximación se degrada a medida que el cambio
relativo se aleja de cero, y lo hace sistemáticamente subestimando el
verdadero cambio porcentual cuando este es positivo y grande. La razón es que
$\log(1+g) \approx g$ solo es una buena aproximación de Taylor de primer orden
para $g$ pequeño; cuando $x$ se duplica, $g=1$ ya no es "pequeño" en ningún
sentido razonable y la aproximación lineal pierde vigencia. **Regla
práctica:** usar la aproximación logarítmica con confianza para cambios de
hasta 10-15% aproximadamente (el error queda por debajo de 1 punto
porcentual); para cambios mayores al 20-30% conviene reportar el cambio
porcentual exacto, o al menos advertir que la aproximación subestima el
crecimiento verdadero.

---
## Ejercicio 1.7 (estructura de datos) — Nivel A

**Enunciado.** Para cada conjunto de datos, indique la estructura (corte
transversal, serie de tiempo, corte transversal combinado o panel).

**a) Precio de cierre diario del dólar mayorista, 2015–2025.**
Serie de tiempo. La unidad de observación es una sola unidad —el tipo de
cambio mayorista— seguida repetidamente a lo largo del tiempo (un valor por
día durante diez años); no hay una sección cruzada de unidades distintas en
ningún momento.

**b) Ingreso, edad y educación de 3,000 hogares relevados por la EPH del
segundo trimestre de 2024.**
Corte transversal. La unidad de observación es el hogar, y todos se observan
en un único momento del tiempo (2T-2024); no hay dimensión temporal.

**c) PBI, población y esperanza de vida de 180 países en 1990, 2000, 2010 y
2020.**
Panel (no necesariamente balanceado). La unidad de observación —el
país— se sigue explícitamente a lo largo de varios momentos del tiempo,
y en cada momento se trata de las mismas 180 unidades, no de muestras nuevas.

**d) Ventas y empleo de 500 empresas industriales, relevadas en 2019 y
nuevamente en 2023, sin garantía de que sean las mismas empresas.**
Corte transversal combinado (*pooled cross section*). Hay dos momentos del
tiempo, pero a diferencia de (c) no hay seguimiento garantizado de las
mismas unidades: son dos muestras independientes de empresas, una por año.
La distinción con panel depende exactamente de ese detalle: "sin garantía de
que sean las mismas empresas" es la pista que descarta panel.

---
## Ejercicio 1.8 (Wooldridge, problema 1.2 — capacitación y productividad) — Nivel A

**Enunciado.** Se le encarga estimar el efecto de la capacitación laboral
sobre la productividad de los trabajadores.

**a) El experimento subyacente.**
El experimento ideal consiste en observar a los mismos trabajadores (o
empresas) en dos estados del mundo idénticos en todo excepto en si reciben
capacitación. Concretamente: tomar un conjunto de trabajadores con igual
capital humano, igual acceso a tecnología y maquinaria, igual calidad de
gestión y bajo las mismas condiciones de mercado, asignar aleatoriamente
a una parte de ellos a recibir el programa de capacitación y a la otra parte
no, y comparar luego su productividad (por ejemplo, producto por hora
trabajada) manteniendo constante todo lo demás. La aleatorización es lo que
garantiza que "todo lo demás" quede balanceado entre ambos grupos, en
promedio, sin necesidad de observarlo ni medirlo.

**b) ¿Es la decisión de capacitar independiente de las características de
los trabajadores?**
En general, no. Las empresas deciden a quién capacitar, y esa decisión rara
vez es aleatoria. Dos características **medibles** que romperían la
independencia: el nivel educativo previo de los trabajadores (empresas con
plantillas más educadas pueden ser las que más invierten en capacitación
adicional) y su antigüedad o experiencia en el puesto. Dos características
**no medibles** que también la romperían: la motivación o habilidad innata
de los trabajadores (los más motivados se ofrecen o son elegidos para
capacitarse) y la calidad de la gestión gerencial de la empresa (una empresa
mejor gestionada puede capacitar más *y*, por esa misma buena gestión, lograr
mayor productividad con independencia de la capacitación).

**c) Un factor ajeno a los trabajadores que afecte la productividad.**
La calidad del capital físico y la tecnología con la que trabajan (maquinaria,
software, la organización del proceso productivo). Dos plantas con
trabajadores idénticos pueden diferir mucho en productividad simplemente
porque una tiene equipamiento más moderno.

**d) ¿Una correlación positiva entre producción y capacitación establece
causalidad de manera convincente?**
No. Es exactamente el problema central del capítulo: la correlación es
compatible con al menos dos historias no causales. Primero, causalidad
inversa: las empresas más rentables o productivas por otras razones (mejor
gestión, mejor tecnología) son las que tienen más recursos ociosos para
destinar a capacitar a su personal, de modo que es la productividad previa la
que genera la capacitación y no al revés. Segundo, una variable omitida como
la calidad gerencial afecta simultáneamente a la probabilidad de capacitar y
al nivel de productividad, generando una correlación positiva entre
capacitación y producción aunque la capacitación en sí no tuviera ningún
efecto causal. Para atribuir la diferencia a la capacitación haría falta, como
en el ejercicio resuelto de `jtrain2`, algo que se aproxime a una asignación
aleatoria (o al menos controlar explícitamente por gestión, tecnología y
características previas de los trabajadores).